In [1]:
from background.middlewares.service_handler import ServiceHandler
from background.modules.exchange_upbit import ExChangeUpbit
from background.modules.log import Log
from background.modules.redis import r as redis_single

from background.dao.research_result_dao import ResearchResultDao
import pandas as pd

import numpy as np
import os
import time
import datetime

In [14]:

class CvResearchProd04Gen01():

    def __init__(
            self,
            tickers
        ):

        self.redis = redis_single
        self.upbit = ExChangeUpbit()

        self.research_result_dao = ResearchResultDao()
        self.tickers = tickers
        self.ticker = None
        
        self.target_days_list = [
            31,
            62,
            93,
            # 186
        ]
    
    def main(self):

        try:
            for ticker in self.tickers:
                self.ticker = ticker
                self.upbit.set_ticker(self.ticker)
                self.decide_hot_cold_time()

        except Exception as e:
            print(f"class : {self.__class__.__name__}, error : {str(e)}")

    def decide_hot_cold_time(self, interval="minute5"):
        
        try:
            count=200
            for days in self.target_days_list:
                
                # 1. 데이터 수집
                to_ = datetime.datetime.now()
                from_ = to_ - datetime.timedelta(days=days)
    
                self.upbit.set_ohlcvs(
                    interval=interval, 
                    days=days, 
                    count=count, 
                    from_=from_, 
                    to_=to_
                )
    
                df = self.upbit.set_volume_imbalance()
                df = self.upbit.set_hour()
                df = self.upbit.set_weekday()
    
                if df is None or len(df) == 0:
                    return
    
                # 2. 기본 컬럼 체크
                required_cols = ["volume", "volume_imbalance", "hour", "weekday"]
    
                for col in required_cols:
                    if col not in df.columns:
                        self.log.info(f"[{self.ticker}] missing column: {col}")
                        return
    
                # 3. 시간대별 집계
                volume_by_hour = df.groupby("hour")["volume"].mean()
                imbalance_by_hour = df.groupby("hour")["volume_imbalance"].mean()
    
                volume_by_weekday = df.groupby("weekday")["volume"].mean()
                imbalance_by_weekday = df.groupby("weekday")["volume_imbalance"].mean()
    
                # 4. HOT TIME 추출
                top_volume_hours = volume_by_hour.sort_values(ascending=False).head(10).index.tolist()
                bot_volume_hours = volume_by_hour.sort_values(ascending=True).head(10).index.tolist()
    
                # 방향성
                best_long_hours = imbalance_by_hour.sort_values(ascending=False).head(10).index.tolist()
                best_short_hours = imbalance_by_hour.sort_values(ascending=True).head(10).index.tolist()
    
                print(f"FOR {days}, top_volume_hours : {top_volume_hours}")
                print(f"FOR {days}, bot_volume_hours : {bot_volume_hours}")
                print(f"FOR {days}, best_long_hours : {best_long_hours}")
                print(f"FOR {days}, best_short_hours : {best_short_hours}")

        except Exception as e:
            print(e)

In [15]:
tickers = ["KRW-ETH"]
cvobj = CvResearchProd04Gen01(tickers)

In [16]:
cvobj.main()

FOR 31, top_volume_hours : [23, 22, 0, 10, 9, 20, 21, 12, 11, 1]
FOR 31, bot_volume_hours : [4, 5, 3, 6, 2, 14, 15, 19, 13, 16]
FOR 31, best_long_hours : [17, 4, 16, 18, 3, 5, 2, 9, 6, 22]
FOR 31, best_short_hours : [14, 7, 20, 10, 13, 23, 0, 8, 11, 19]
FOR 62, top_volume_hours : [0, 23, 9, 22, 1, 10, 21, 20, 12, 8]
FOR 62, bot_volume_hours : [4, 5, 6, 14, 19, 3, 2, 15, 13, 7]
FOR 62, best_long_hours : [5, 17, 9, 3, 12, 6, 4, 2, 15, 16]
FOR 62, best_short_hours : [13, 14, 11, 10, 7, 0, 19, 18, 23, 22]
FOR 93, top_volume_hours : [23, 0, 9, 22, 10, 1, 21, 11, 8, 20]
FOR 93, bot_volume_hours : [4, 5, 6, 14, 3, 19, 2, 13, 16, 15]
FOR 93, best_long_hours : [5, 17, 9, 6, 4, 16, 3, 12, 15, 0]
FOR 93, best_short_hours : [13, 10, 14, 11, 22, 2, 7, 21, 23, 19]
